In [0]:
-- Clean columns names

-- Find Duplicates


-- Data Transformations and Final Query
With
manufacture_year as
(
  Select 
    year as year_of_manufacture,
    Case 
      When year < 2000 then 'Pre 2000'
      else 'Post 2000'
      End as Manufacture_Year_Category
  From car_sales_data 
),
date_of_sale as
( 
  with string_to_date as 
    (
      Select
        to_date(SUBSTR(saledate, 5, 11), 'MMM dd yyyy') as date_sold
      from car_sales_data 
    )
    Select
    date_sold,
    Extract(DAY from date_sold)  AS day_of_month,  -- 1 - 31
    date_format(date_sold, 'MMM') AS month_name,   -- Jan - Dec
    date_format(date_sold, 'EEE') AS weekday_name,  -- Mon - Sun
    Extract(YEAR from date_sold) as year_sold
    from string_to_date, manufacture_year
),
vehicle_age_at_sale as
(
Select   --Extract(YEAR from date_sold) - year_of_manufacture as age_at_sale
  year_sold - year_of_manufacture as age_at_sale
  from date_of_sale, manufacture_year
),
vehicle_mileage as
(
  Select 
      Case
        when odometer = 1 then 'Investigate'
        when odometer > 1 and odometer <= 150000 then 'Low Mileage'
        when odometer > 150000 and odometer <= 250000 then 'Moderate Mileage'
        else 'High Mileage'
      End as mileage_classification 
  from car_sales_data 
),
vehicle_condition as
(
  Select 
    Case  
      when condition in (1,10)  then 'very bad'
      when condition in (11-20) then 'poor'
      when condition in (21,30) then 'fair'
      when condition in (31-40) then 'good'
    Else 'excellent'
    End As condition_buckets,
      -- Calculate KM per Year (Safe division to avoid 0 age error)
    (odometer / NULLIF(age_at_sale, 0)) AS km_per_year,
    Case  
      when (odometer / NULLIF(age_at_sale, 0)) <= 10000 THEN 'Pristine'
      when (odometer / NULLIF(age_at_sale, 0)) <= 20000 THEN 'Standard'
      when (odometer / NULLIF(age_at_sale, 0)) <= 35000 THEN 'Well-Used'
      when (odometer / NULLIF(age_at_sale, 0)) <= 60000 THEN 'Heavy Wear'
      Else 'Abused / Extreme'
      End As mileage_per_year
    from car_sales_data, vehicle_mileage, date_of_sale, vehicle_age_at_sale
),
Pricing as
(
  Select
      mmr as cost_price,
      sellingprice as sell_price,
      sell_price - mmr as gross_profit,
      100*(sell_price - mmr)/sell_price as gross_profit_margin,
    Case 
      when gross_profit_margin <= 0 then 'Loss'
      when gross_profit_margin > 0 and gross_profit_margin <= 5 then 'Good'
      when gross_profit_margin > 5 and gross_profit_margin <= 8 then 'high'
      Else 'Excellent'
      End As profit_margin_classification
  from car_sales_data 
)

Select
-- vehicle description columns
  make,
  model,
  concat(make,' ',model) as make_model,
  trim,
  body,
  transmission,

-- date columns
  year_of_manufacture,
  Manufacture_Year_Category,
  date_sold,
  day_of_month,  -- 1 - 31
  month_name,   -- Jan - Dec
  weekday_name,  -- Mon - Sun
  year_sold,
  age_at_sale,

-- condition columns
  condition,
  condition_buckets,
  mileage_per_year,
  mileage_classification,

-- pricing columns
  gross_profit,
  gross_profit_margin
  
from 
  car_sales_data, 
  date_of_sale,
  manufacture_year,
  Pricing,
  vehicle_condition,
  vehicle_mileage,
  vehicle_age_at_sale